# Train a Tabular Regressor — XGBoost (simple)

**Maintained by:** IGNODE  
**Last verified:** 2026-05-30 against XGBoost 2.1, scikit-learn 1.5  
**Runtime:** under 1 minute on Colab's free CPU tier

Train a tabular regression model using **XGBoost**. Linear notebook (no branching, no AutoML) — read it end-to-end. If you want algorithm comparison or AutoML, use `train_tabular_regression.ipynb`.

## Output

`model.onnx` + `feature_columns.json` — drop into IGNODE → ML Factory → Custom Models → + Upload ML Model.

## Quick start

### To try it right now (no setup needed)

1. **Runtime → Run all** at the top of Colab
2. Wait ~30 seconds for dependencies + ~10 seconds for training
3. The last cell automatically downloads `model.onnx` + the sidecar JSON to your laptop — that's a trained XGBoost regressor on the sample IoT data

### To train on YOUR data

You only need to edit **two values**:

| Step | Cell | What to change |
|---|---|---|
| 1 | **Load data** cell (below) | `SAMPLE_DATASET = 'equipment_rul_regression'` → `SAMPLE_DATASET = None` |
| 2 | **Settings** cell | `LABEL_COLUMN = 'RemainingLife'` → `LABEL_COLUMN = 'your_numeric_target_column'` |

The target column **must be numeric**.

### What you get at the end

- `model.onnx` — your trained regression model
- `feature_columns.json` — input contract (`{feature_columns, label_columns}`)

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

## 1. Install pinned dependencies

In [ ]:
!pip install --quiet \
    xgboost==2.1.2 \
    scikit-learn==1.5.2 \
    onnx==1.21.0 \
    onnxmltools==1.16.0 \
    onnxconverter-common==1.14.0

import xgboost as xgb
print(f'XGBoost: {xgb.__version__}')

## 2. Load data

Ships set to a small IoT sample (`equipment_rul_regression.csv`) so the notebook runs end-to-end out of the box. To use your own CSV, set `SAMPLE_DATASET = None` below and update `LABEL_COLUMN` in the Settings cell to your numeric target column.

In [ ]:
# ───────── EDIT THIS ─────────
SAMPLE_DATASET = 'equipment_rul_regression'   # set to None to upload your own CSV
# Available samples (regression):
#   'equipment_rul_regression'   — equipment telemetry, label='RemainingLife'
#   'building_energy_regression' — building features, label='HeatingLoad'
# ────────────────────────────

import pandas as pd

if SAMPLE_DATASET:
    url = f'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/{SAMPLE_DATASET}.csv'
    df = pd.read_csv(url)
    print(f'Loaded sample {SAMPLE_DATASET!r}: {df.shape[0]} rows x {df.shape[1]} columns')
else:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))
    df = pd.read_csv(csv_path)
    print(f'Loaded {csv_path}: {df.shape[0]} rows x {df.shape[1]} columns')

display(df.head())

## 3. Settings

In [ ]:
# ───────── EDIT THESE ─────────
LABEL_COLUMN = 'RemainingLife'   # sample default; change when you bring your own CSV

N_ESTIMATORS = 200
LEARNING_RATE = 0.05
MAX_DEPTH = 6

TEST_SIZE = 0.2
RANDOM_SEED = 42
# ──────────────────────────────

if LABEL_COLUMN not in df.columns:
    raise ValueError(f"Label column '{LABEL_COLUMN}' not in CSV. Available: {list(df.columns)}")
if not pd.api.types.is_numeric_dtype(df[LABEL_COLUMN]):
    raise ValueError(
        f"Label column '{LABEL_COLUMN}' must be numeric for regression. "
        f"Got dtype {df[LABEL_COLUMN].dtype}. Use the classifier notebook instead."
    )

## 4. Prep the data

In [ ]:
import re
from sklearn.model_selection import train_test_split

def normalize(name):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')

rename_map = {c: normalize(c) for c in df.columns if c != normalize(c)}
if rename_map:
    print('Renamed columns:')
    for old, new in rename_map.items():
        print(f'  {old!r}  ->  {new!r}')
    df = df.rename(columns=rename_map)
    if LABEL_COLUMN in rename_map:
        LABEL_COLUMN = rename_map[LABEL_COLUMN]

X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN].astype(float).values
feature_columns = list(X.columns)

print(f'Features ({len(feature_columns)}): {feature_columns}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f'Train: {X_train.shape[0]} rows. Test: {X_test.shape[0]} rows.')

## 5. Train

In [ ]:
import time

t0 = time.time()
model = xgb.XGBRegressor(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)
model.fit(X_train, y_train)
print(f'Trained in {time.time() - t0:.1f} sec')

## 6. Evaluate

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Test-set RMSE: {rmse:.3f}')
print(f'Test-set MAE:  {mae:.3f}')
print(f'Test-set R²:   {r2:.3f}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.6, s=20)
lim = (min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max()))
ax.plot(lim, lim, 'r--', linewidth=1, label='perfect prediction')
ax.set_xlabel(f'Actual {LABEL_COLUMN}')
ax.set_ylabel(f'Predicted {LABEL_COLUMN}')
ax.set_title(f'Predicted vs Actual (test set, R²={r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Export to ONNX

In [ ]:
from onnxmltools.convert import convert_xgboost
from onnxconverter_common.data_types import FloatTensorType
import onnx

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]

onnx_model = None
for opset in (18, 15, 12):
    try:
        onnx_model = convert_xgboost(model, initial_types=initial_types, target_opset=opset)
        print(f'Exported at opset {opset}')
        break
    except Exception as ex:
        print(f'  opset {opset} failed ({type(ex).__name__}); trying lower')
if onnx_model is None:
    raise RuntimeError('All opset attempts failed.')

onnx.save_model(onnx_model, 'model.onnx')
print(f'Saved model.onnx ({len(onnx_model.SerializeToString()) / 1024:.1f} KB)')

## 8. Write sidecar file

In [ ]:
import json

feature_sidecar = {'feature_columns': feature_columns, 'label_columns': [LABEL_COLUMN]}
with open('feature_columns.json', 'w') as f:
    json.dump(feature_sidecar, f, indent=2)

print('feature_columns.json:')
print(json.dumps(feature_sidecar, indent=2))

## 9. Download

In [ ]:
from google.colab import files
files.download('model.onnx')
files.download('feature_columns.json')

## 10. Upload to IGNODE

1. **Integrations → ML Factory** in your IGNODE portal
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model**
4. Drop your `model.onnx` and fill in the metadata form:
    - **Task Type:** `Regression`
    - **Feature Columns:** paste from `feature_columns.json` → `feature_columns`
    - **Target Column:** paste from `feature_columns.json` → `label_columns` (single entry)
5. Click **Upload**, then **Open in Playground** to test.

---

## Reusing this notebook for your own data

Two edits switch to your own dataset:

```python
# In the "Load data" cell:
SAMPLE_DATASET = None        # was 'equipment_rul_regression'

# In the "Settings" cell:
LABEL_COLUMN = 'YourColumn'  # was 'RemainingLife' — your NUMERIC target column
```

Everything else adapts automatically. The target column must be numeric — strings throw a clear error pointing at the classifier notebook.

### Optional tweaks

| Want to change | Edit (Settings cell) |
|---|---|
| Algorithm strength | `N_ESTIMATORS = 500` |
| Step size | `LEARNING_RATE = 0.01` |
| Tree depth | `MAX_DEPTH = 8` |
| Train/test ratio | `TEST_SIZE = 0.3` |
| Reproducibility | `RANDOM_SEED = <any int>` |

### Common errors

| Error | Fix |
|---|---|
| `Label column must be numeric for regression` | Use the classifier notebook instead |
| `Label column 'X' not in CSV` | Check the column list printed by the inspect cell |
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns shown in the prep cell |
| Predictions look biased | Try the AutoML notebook (`train_tabular_regression.ipynb`) for algorithm comparison |